In [1]:
# === Ô GHI LOG (mẫu PL1 bản 2 – chạy được với ipykernel 7): mọi thứ in ra màn hình đồng thời được ghi vào file log ===
import logging, sys, datetime, warnings
from pathlib import Path
TEN_NOTEBOOK = "07_nhap_wandb_experiment_log_Han"
NGUOI_CHAY   = "Han"

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())  # thư mục gốc repo
thu_muc_log = ROOT / "logs" / "nhap_wandb_log"
thu_muc_log.mkdir(parents=True, exist_ok=True)
file_log = thu_muc_log / f"{datetime.datetime.now():%Y-%m-%d_%H%M}_{TEN_NOTEBOOK}.log"
tep_log = open(file_log, "a", encoding="utf-8", buffering=1)

if hasattr(sys.stdout, "echo"):                  # chạy trong Jupyter / VS Code: nhờ kernel chép thêm ra file, không thay sys.stdout
    sys.stdout.echo = tep_log
    sys.stderr.echo = tep_log
else:                                            # chạy bằng python thường
    class GhiCaHai:
        def __init__(self, man_hinh, tep): self.man_hinh, self.tep = man_hinh, tep
        def write(self, s): self.man_hinh.write(s); self.tep.write(s); return len(s)
        def flush(self): self.man_hinh.flush(); self.tep.flush()
        def __getattr__(self, ten): return getattr(self.man_hinh, ten)
    sys.stdout = GhiCaHai(sys.__stdout__, tep_log)

logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True,
                    format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logging.captureWarnings(True)
log = logging.getLogger(TEN_NOTEBOOK)
log.info(f"Bắt đầu chạy {TEN_NOTEBOOK} – người chạy: {NGUOI_CHAY} – file log: {file_log}")

2026-11-14 00:07:28 | INFO | Bắt đầu chạy 07_nhap_wandb_experiment_log_Han – người chạy: Han – file log: /Users/caongochan/Documents/Nhom11_DuBaoDienNang_GiuaKy/logs/2026-11-14_0007_07_nhap_wandb_experiment_log_Han.log


# 07 – Nhập kết quả mô hình từ `logs/experiment_log.xlsx` lên W&B

| | |
|---|---|
| **Tác giả** | Cao Ngọc Hân – Nhóm 11 |
| **Ngày tạo** | 14/11/2026 |
| **Mục đích** | Đưa các dòng EXP-001 → EXP-022 (mô hình của Linh và Nhung, chạy 01–03/11) lên W&B project `cuoi-ky-du-bao-dien-nang`, để có minh chứng W&B giống 2 challenge. |
| **Đọc vào** | `logs/experiment_log.xlsx`, sheet `Experiment_Log` |
| **Ghi ra** | Run W&B (mỗi dòng EXP một run), `logs/wandb_links_experiment_log.csv` |

**Lưu ý quan trọng:** notebook này **không train lại mô hình và không chạy lại Test**. Nó chỉ chép số đã có trong file Excel lên W&B. Mỗi run ghi rõ ngày chạy gốc, người chạy gốc, và có tag `nhap_tu_experiment_log`. Các dòng có ghi chú "SỐ CŨ" (EXP-007 → EXP-010) có thêm tag `so_cu` để không ai lấy nhầm.

**Cách chạy:** mặc định chỉ in danh sách run sẽ tạo (chạy thử). Muốn ghi thật lên W&B, chạy **một lần**:
`BAT_WANDB=1 jupyter nbconvert --to notebook --execute --inplace notebooks/07_nhap_wandb_experiment_log_Han.ipynb`.
Chạy lại thì các ID đã có trong `wandb_links_experiment_log.csv` sẽ được bỏ qua, không tạo run trùng.

In [2]:
import os, unicodedata, re
import pandas as pd

USE_WANDB = os.environ.get("BAT_WANDB") == "1"
FILE_EXCEL = ROOT / "logs" / "experiment_log.xlsx"
FILE_LINK = ROOT / "logs" / "wandb_links_experiment_log.csv"

bang = pd.read_excel(FILE_EXCEL, sheet_name="Experiment_Log")
bang = bang[bang["ID"].astype(str).str.startswith("EXP-")].reset_index(drop=True)
log.info(f"Đọc {len(bang)} dòng EXP từ {FILE_EXCEL.name}: {', '.join(bang['ID'])}")

def khong_dau(s):
    s = unicodedata.normalize("NFD", str(s)).replace("đ", "d").replace("Đ", "D")
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-zA-Z0-9]+", "-", s).strip("-").lower()

bang["ten_run"] = [f"ck-{r['ID']}-{khong_dau(r['Người chạy'])}-{khong_dau(r['Mô hình'])[:30]}-{khong_dau(r['Tập đánh giá'])}" for _, r in bang.iterrows()]
bang["so_cu"] = bang["Nhận xét"].astype(str).str.contains("SỐ CŨ")
da_co = set(pd.read_csv(FILE_LINK)["ID"]) if FILE_LINK.exists() else set()
print(bang[["ID", "Ngày", "Người chạy", "Mô hình", "Tập đánh giá", "RMSE", "ten_run", "so_cu"]].to_string())
log.info(f"Đã có trên W&B từ trước: {sorted(da_co) or 'chưa có'}")

2026-11-14 00:07:28 | INFO | Đọc 21 dòng EXP từ experiment_log.xlsx: EXP-001, EXP-002, EXP-003, EXP-004, EXP-005, EXP-006, EXP-007, EXP-008, EXP-009, EXP-010, EXP-011, EXP-012, EXP-013, EXP-015, EXP-016, EXP-017, EXP-018, EXP-019, EXP-020, EXP-021, EXP-022


         ID       Ngày Người chạy                                        Mô hình Tập đánh giá      RMSE                                                   ten_run  so_cu
0   EXP-001 2026-11-01       Linh                                      Naive t-1     CV-Train  0.684100                        ck-EXP-001-linh-naive-t-1-cv-train  False
1   EXP-002 2026-11-01       Linh                                      Naive t-1        Train  0.695500                           ck-EXP-002-linh-naive-t-1-train  False
2   EXP-003 2026-11-01       Linh                                     Naive t-24     CV-Train  0.941400                       ck-EXP-003-linh-naive-t-24-cv-train  False
3   EXP-004 2026-11-01       Linh                                     Naive t-24        Train  0.985100                          ck-EXP-004-linh-naive-t-24-train  False
4   EXP-005 2026-11-01       Linh                              Linear Regression     CV-Train  0.593300                ck-EXP-005-linh-linear-regression-cv

In [3]:
if not USE_WANDB:
    log.info("CHẠY THỬ: chưa ghi lên W&B. Muốn ghi thật thì chạy lại với BAT_WANDB=1 (xem ô trên).")
else:
    import wandb, datetime
    links = []
    for _, r in bang.iterrows():
        if r["ID"] in da_co:
            log.info(f"Bỏ qua {r['ID']} (đã có run)"); continue
        run = wandb.init(
            entity="sgu-mayhoc-nhom11", project="cuoi-ky-du-bao-dien-nang", name=r["ten_run"],
            tags=[r["ID"], str(r["Người chạy"]), str(r["Tập đánh giá"]), "nhap_tu_experiment_log"] + (["so_cu"] if r["so_cu"] else []),
            config={"ID": r["ID"], "ngay_chay_goc": str(pd.Timestamp(r["Ngày"]).date()), "nguoi_chay_goc": r["Người chạy"],
                    "mo_hinh": r["Mô hình"], "bo_dac_trung": r["Bộ đặc trưng"], "cach_chia": r["Cách chia dữ liệu"],
                    "tham_so": str(r["Tham số"]), "tap_danh_gia": r["Tập đánh giá"], "nhan_xet_goc": r["Nhận xét"],
                    "cach_ghi": f"Nhập lại từ logs/experiment_log.xlsx ngày {datetime.date.today():%d/%m/%Y} bởi Hân; không train lại, không chạy lại Test"},
            reinit=True)
        run.summary.update({"RMSE": float(r["RMSE"]), "MAE": float(r["MAE"]), "R2": float(r["R²"]),
                            "thoi_gian_chay_s": float(r["Thời gian chạy (s)"] or 0)})
        links.append({"ID": r["ID"], "ten_run": r["ten_run"], "url": run.url})
        log.info(f"{r['ID']} → {run.url}")
        run.finish()
    if links:
        moi = pd.DataFrame(links)
        tat_ca = pd.concat([pd.read_csv(FILE_LINK), moi]) if FILE_LINK.exists() else moi
        tat_ca.to_csv(FILE_LINK, index=False, encoding="utf-8-sig")
        log.info(f"Đã ghi {len(links)} run mới; danh sách link ở {FILE_LINK.relative_to(ROOT)}")

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/caongochan/.netrc.


wandb: Currently logged in as: caongochan142 (sgu-mayhoc-nhom11) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: setting up run tb3shrt7


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /Users/caongochan/Documents/Nhom11_DuBaoDienNang_GiuaKy/notebooks/wandb/run-20261114_000730-tb3shrt7
wandb: Run `wandb offline` to turn off syncing.


... (286 outputs left out)

wandb: 
wandb: Run summary:
wandb:              MAE 0.3292
wandb:               R2 0.6504
wandb:             RMSE 0.475
wandb: thoi_gian_chay_s 0.0231
wandb: 


wandb: 🚀 View run ck-EXP-022-nhung-xgboost-toi-uu-test at: https://forge.coreweave.com/wandb/sgu-mayhoc-nhom11/cuoi-ky-du-bao-dien-nang/runs/ao1hhdu0
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/sgu-mayhoc-nhom11/cuoi-ky-du-bao-dien-nang
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261114_000926-ao1hhdu0/logs


2026-11-14 00:09:30 | INFO | Đã ghi 21 run mới; danh sách link ở logs/wandb_links_experiment_log.csv


## Nhật ký

- 14/11/2026 – Tạo notebook nhập 21 dòng EXP từ `logs/experiment_log.xlsx` lên W&B (không train lại, không chạy lại Test). Commit `...`.